# 🔬 Retina Insight: Rigorous Research Training & Multi-Cohort Evaluation Pipeline
### Enhanced Deep Learning Framework for Diabetic Retinopathy Grading

**Publication-Ready Training Protocol:**
- **Datasets:**
  - **Internal Development:** APTOS 2019 Blindness Detection (3,662 fundus photographs)
  - **External Independent Test:** IDRiD (Indian Diabetic Retinopathy Image Dataset - 103 test scans)
- **Experimental Rigor:**
  - **3-Way Stratified Partition:** 70% Train / 15% Validation / 15% Strict Held-Out Test (zero data leakage)
  - **Domain Shift Mitigation:** Random resized cropping, scale jitter (80%-105%), aspect framing, rotation, and color illumination jitter
  - **Extended Training:** 15 epochs with Cosine Annealing learning rate schedule and AdamW
  - **Class Rebalancing:** Effective class-frequency weighted cross-entropy
  - **Uncertainty Calibration:** Post-hoc Temperature Scaling ($T$) fitted strictly on validation set and verified on held-out test
  - **Clinical Decision Tiers:** Quadratic Weighted Kappa (QWK), Expected Calibration Error (ECE), and Referable DR (RDR) ROC-AUC


## 1. Setup, Package Ingestion & GPU Environment


In [ ]:
import os
import cv2
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Execution Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"   Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Ingestion & Strict 3-Way Stratified Split (Train / Val / Held-Out Test)
To eliminate optimistic validation bias, the APTOS dataset is partitioned into:
- **Train (70%):** Model parameter optimization
- **Validation (15%):** Hyperparameter tracking, checkpointing & Temperature Scaling fitting
- **Internal Held-Out Test (15%):** Untouched during training and calibration; evaluated only once at the conclusion


In [ ]:
# -------------------------------------------------------------------------
# Direct Discovery & Verification of APTOS train.csv and Images (Handles Zip)
# -------------------------------------------------------------------------
import zipfile

print("🔍 Scanning /kaggle/input for APTOS training dataset...")
all_files = []
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    for f in files:
        all_files.append(os.path.join(root, f))

# 1. Find CSV files
csv_files = [f for f in all_files if f.lower().endswith('.csv')]
csv_path = None

# Priority: Exact 'train.csv' not in IDRiD
for c in csv_files:
    if os.path.basename(c).lower() == 'train.csv' and 'idrid' not in c.lower():
        csv_path = c
        break

if csv_path is None:
    for c in csv_files:
        fname = os.path.basename(c).lower()
        if 'idrid' not in c.lower() and not any(bad in fname for bad in ['sample', 'submission', 'test']):
            csv_path = c
            break

if csv_path is None:
    raise FileNotFoundError("Could not locate APTOS train.csv.")

print(f"✅ Selected Training CSV: {csv_path}")
aptos_dir = os.path.dirname(csv_path)

# 2. Find Image Directory (Check for existing uncompressed images OR zip archive)
image_dir = None

# Check subfolders inside aptos_dir for images first
for root, dirs, files in os.walk(aptos_dir, followlinks=True):
    if any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in files):
        image_dir = root
        break

# If no uncompressed images in aptos_dir, check for zip archives (e.g. train.zip)
if image_dir is None:
    zip_files = [f for f in os.listdir(aptos_dir) if f.lower().endswith('.zip') and ('train' in f.lower() or 'image' in f.lower())]
    if not zip_files:
        zip_files = [f for f in os.listdir(aptos_dir) if f.lower().endswith('.zip')]
        
    if zip_files:
        target_zip = os.path.join(aptos_dir, zip_files[0])
        extract_dest = '/tmp/aptos_train_images'
        if not os.path.exists(extract_dest) or len(os.listdir(extract_dest)) == 0:
            os.makedirs(extract_dest, exist_ok=True)
            print(f"📦 Found archive: {zip_files[0]} ({os.path.getsize(target_zip)/(1024*1024):.1f} MB)")
            print(f"   Extracting to {extract_dest} (fast extraction to /tmp)... Please wait 15-30s.")
            with zipfile.ZipFile(target_zip, 'r') as z:
                z.extractall(extract_dest)
            print("   Extraction complete!")
        else:
            print(f"✅ Extracted images already cached at: {extract_dest}")
            
        # Check if extracted into an inner folder
        sub_dirs = [os.path.join(extract_dest, d) for d in os.listdir(extract_dest) if os.path.isdir(os.path.join(extract_dest, d))]
        if sub_dirs and any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in os.listdir(sub_dirs[0])):
            image_dir = sub_dirs[0]
        else:
            image_dir = extract_dest

if image_dir is None:
    print(f"\n⚠️ Files in {aptos_dir}:")
    for f in os.listdir(aptos_dir):
        print(f"   - {f}")
    raise FileNotFoundError("No images or zip archives found inside the APTOS folder.")

# Verify images in image_dir
found_img_count = len([f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
print(f"✅ Target Image Folder:  {image_dir} (Verified {found_img_count} images)")

df_full = pd.read_csv(csv_path)
diag_col = next(c for c in df_full.columns if any(k in c.lower() for k in ['diag', 'grade']))
id_col = next(c for c in df_full.columns if any(k in c.lower() for k in ['id_code', 'id', 'image']))
df_full = df_full.rename(columns={diag_col: 'diagnosis', id_col: 'id_code'})
df_full['diagnosis'] = df_full['diagnosis'].astype(int)

print(f"\n📊 Loaded {len(df_full)} Scans across 5 DR Stages:")
for s, count in df_full['diagnosis'].value_counts().sort_index().items():
    print(f"     Stage {s}: {count:>4} scans ({count/len(df_full)*100:.1f}%)")

# 1. First Split: 70% Train, 30% Temp (Val + Test)
train_df, temp_df = train_test_split(
    df_full,
    test_size=0.30,
    random_state=SEED,
    stratify=df_full['diagnosis']
)

# 2. Second Split: Split Temp evenly into 15% Val and 15% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['diagnosis']
)

print(f"\n📊 Strict 3-Way Stratified Partition:")
print(f"   Train set:     {len(train_df):>5} images ({len(train_df)/len(df_full)*100:.1f}%)")
print(f"   Val set:       {len(val_df):>5} images ({len(val_df)/len(df_full)*100:.1f}%) [for checkpointing & temp fit]")
print(f"   Held-out Test: {len(test_df):>5} images ({len(test_df)/len(df_full)*100:.1f}%) [strictly untouched]")


## 3. Preprocessing & Domain-Generalizable Augmentation Pipeline
To combat **domain shift** (camera sensors, framing, and zoom variations across clinics):
1. **Ben Graham Color Contrast Subtraction:** Standardizes illumination and emphasizes microaneurysms and exudates ($4 \times \text{img} - 4 \times \text{blur} + 128$).
2. **Framing & Scale Jitter:** `RandomResizedCrop` with scale $(0.80, 1.05)$ teaches the model scale-invariance.
3. **Color Jitter:** Subtle brightness and contrast variations simulate inter-camera exposure variance.


In [ ]:
def fast_ben_graham(img_bgr, target_size=256):
    img_resized = cv2.resize(img_bgr, (target_size, target_size))
    blurred = cv2.GaussianBlur(img_resized, (0, 0), 10)
    enhanced = cv2.addWeighted(img_resized, 4, blurred, -4, 128)
    return enhanced

class RobustRetinaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row['id_code'])
        
        # Check extensions
        img_path = None
        for ext in ['', '.png', '.jpg', '.jpeg']:
            p = os.path.join(self.img_dir, f"{img_id}{ext}")
            if os.path.exists(p):
                img_path = p
                break
                
        if img_path and os.path.exists(img_path):
            img_bgr = cv2.imread(img_path)
        else:
            img_bgr = np.zeros((256, 256, 3), dtype=np.uint8)
            
        if img_bgr is None:
            img_bgr = np.zeros((256, 256, 3), dtype=np.uint8)
            
        enhanced_bgr = fast_ben_graham(img_bgr, target_size=256)
        enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(enhanced_rgb)
        
        if self.transform:
            tensor_img = self.transform(pil_img)
        else:
            tensor_img = transforms.ToTensor()(pil_img)
            
        label = torch.tensor(row['diagnosis'], dtype=torch.long)
        return tensor_img, label

# Training Transforms: Scale Jitter + Framing + Rotations + Color Jitter
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(256, scale=(0.80, 1.05), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# Evaluation Transforms (Standardized deterministic resize & normalization)
eval_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = RobustRetinaDataset(train_df, image_dir, transform=train_transforms)
val_dataset = RobustRetinaDataset(val_df, image_dir, transform=eval_transforms)
test_dataset = RobustRetinaDataset(test_df, image_dir, transform=eval_transforms)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("✅ DataLoaders initialized with framing & scale augmentation.")


## 4. Deep Architecture: EfficientNet-B0 with Effective Class-Frequency Weighting
We apply transfer learning with pre-trained ImageNet weights. Class weights are derived inversely proportional to class frequencies to ensure rare sight-threatening lesions (Stage 3 and 4) exert adequate gradient influence.


In [ ]:
# 1. Compute 5-Class Balanced Weights (Ensures exactly 5 weights for stages 0-4)
ALL_CLASSES = np.array([0, 1, 2, 3, 4])
weights = compute_class_weight(
    class_weight='balanced',
    classes=ALL_CLASSES,
    y=train_df['diagnosis'].values
)
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)
print(f"✅ 5-Class Weights Tensor (shape: {list(class_weights_tensor.shape)}): {[round(w, 2) for w in weights]}")

# 2. EfficientNet-B0 Backbone
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.35),
    nn.Linear(in_features, 5)
)
model = model.to(device)

# 3. Loss, Optimizer & Cosine Annealing Schedule (15 Epochs)
EPOCHS = 15
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=2.5e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print("✅ EfficientNet-B0 configured with 5-class weighted loss and Cosine Annealing.")


## 5. Extended 15-Epoch Training Loop with Validation Checkpointing
Trains for 15 epochs, logging loss, accuracy, and Quadratic Weighted Kappa (QWK) per epoch, saving the checkpoint with the highest validation QWK.


In [ ]:
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'val_qwk': []
}

best_val_qwk = -1.0
CHECKPOINT_PATH = '/kaggle/working/best_retina_model_paper.pth'

print(f"🚀 Commencing {EPOCHS}-Epoch Training Run...\n")

for epoch in range(1, EPOCHS + 1):
    # ------------ TRAINING PHASE ------------
    model.train()
    train_loss, train_corrects, total_train = 0.0, 0, 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]")
    
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_corrects += torch.sum(preds == labels.data).item()
        total_train += labels.size(0)
        
        train_bar.set_postfix({
            'loss': f"{loss.item():.3f}",
            'acc': f"{train_corrects/total_train:.3f}"
        })
        
    scheduler.step()
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = train_corrects / total_train
    
    # ------------ VALIDATION PHASE ------------
    model.eval()
    val_loss, total_val = 0.0, 0
    val_preds, val_labels = [], []
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Val]")
    
    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
            total_val += labels.size(0)
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = accuracy_score(val_labels, val_preds)
    epoch_val_qwk = cohen_kappa_score(val_labels, val_preds, weights='quadratic')
    
    # Record history
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_acc'].append(epoch_val_acc)
    history['val_qwk'].append(epoch_val_qwk)
    
    current_lr = scheduler.get_last_lr()[0]
    print(f"📊 Epoch {epoch:02d} (lr={current_lr:.2e}):")
    print(f"   Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.3f}")
    print(f"   Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.3f} | Val QWK: {epoch_val_qwk:.4f}")
    
    if epoch_val_qwk > best_val_qwk:
        best_val_qwk = epoch_val_qwk
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"   ⭐ Checkpoint Updated! (Best Val QWK: {best_val_qwk:.4f})\n")

print(f"\n🎉 Training Complete! Peak Validation QWK: {best_val_qwk:.4f}")


## 6. Uncertainty Calibration: Temperature Scaling on Validation Logits
Optimizes temperature scalar $T$ using L-BFGS to minimize negative log-likelihood on the **Validation** set only. Evaluates Expected Calibration Error (ECE) reduction.


In [ ]:
def compute_ece(probs, labels, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == labels

    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(accuracies[in_bin])
            avg_confidence_in_bin = np.mean(confidences[in_bin])
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return ece

# Load best validation model
model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()

# Collect validation logits
val_logits_list, val_targets_list = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        logits = model(images)
        val_logits_list.append(logits)
        val_targets_list.append(labels)

val_logits = torch.cat(val_logits_list).to(device)
val_targets = torch.cat(val_targets_list).to(device)

# Uncalibrated Validation ECE
uncal_val_probs = torch.softmax(val_logits, dim=1).cpu().numpy()
val_ece_before = compute_ece(uncal_val_probs, val_targets.cpu().numpy())

# Optimize Temperature T
temperature = nn.Parameter(torch.ones(1, device=device) * 1.5)
nll_criterion = nn.CrossEntropyLoss()
temp_optimizer = optim.LBFGS([temperature], lr=0.01, max_iter=50)

def eval_closure():
    temp_optimizer.zero_grad()
    loss = nll_criterion(val_logits / temperature, val_targets)
    loss.backward()
    return loss

temp_optimizer.step(eval_closure)
optimal_T = float(temperature.item())

# Calibrated Validation ECE
cal_val_probs = torch.softmax(val_logits / optimal_T, dim=1).cpu().numpy()
val_ece_after = compute_ece(cal_val_probs, val_targets.cpu().numpy())

print("="*60)
print("📊 VALIDATION CALIBRATION RESULTS:")
print(f"   Fitted Temperature (T):        {optimal_T:.4f}")
print(f"   Validation ECE (Uncalibrated): {val_ece_before * 100:.2f}%")
print(f"   Validation ECE (Calibrated):   {val_ece_after * 100:.2f}%")
print("="*60)

with open('/kaggle/working/temperature_paper.txt', 'w') as f:
    f.write(str(optimal_T))


## 7. Unbiased Evaluation on Strict Held-Out APTOS Test Set
Evaluates the final calibrated model on the **15% held-out test cohort** (images never seen during training, checkpointing, or temperature optimization).


In [ ]:
test_logits_list, test_targets_list = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        test_logits_list.append(logits)
        test_targets_list.append(labels)

test_logits = torch.cat(test_logits_list).to(device)
test_targets = torch.cat(test_targets_list).cpu().numpy()

# Calibrated test probabilities
test_probs = torch.softmax(test_logits / optimal_T, dim=1).cpu().numpy()
test_preds = np.argmax(test_probs, axis=1)

# Metrics
test_acc = accuracy_score(test_targets, test_preds)
test_qwk = cohen_kappa_score(test_targets, test_preds, weights='quadratic')
test_ece = compute_ece(test_probs, test_targets)
within_one = np.mean(np.abs(test_targets - test_preds) <= 1)

# Referable DR (Stage >= 2) binary metrics
rdr_targets = (test_targets >= 2).astype(int)
rdr_probs = test_probs[:, 2:].sum(axis=1)
rdr_auc = roc_auc_score(rdr_targets, rdr_probs)

print("="*60)
print("🏆 UNBIASED HELD-OUT APTOS TEST RESULTS:")
print(f"   Test Accuracy:                  {test_acc * 100:.2f}%")
print(f"   Within-One-Stage Accuracy:      {within_one * 100:.2f}%")
print(f"   Quadratic Weighted Kappa (QWK): {test_qwk:.4f}")
print(f"   Calibrated ECE:                 {test_ece * 100:.2f}%")
print(f"   Referable DR (RDR) ROC-AUC:     {rdr_auc:.4f}")
print("="*60)

print("\nClassification Report (Held-Out Test):")
print(classification_report(test_targets, test_preds, target_names=[labels_map[i] for i in range(5)]))


## 8. External Multi-Center Evaluation: IDRiD Test Cohort
Evaluates the model on the official **Indian Diabetic Retinopathy Image Dataset (IDRiD)** test split (103 scans captured at a different clinic with a different camera system) to test out-of-distribution generalizability.


In [ ]:
# -------------------------------------------------------------------------
# Dynamic Discovery & Evaluation on IDRiD External Cohort
# -------------------------------------------------------------------------
print("🔍 Searching for IDRiD Dataset in /kaggle/input...")
idrid_csv = None
idrid_img_dir = None

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.csv') and any(k in f.lower() for k in ['idrid', 'grading', 'clinical', 'labels']):
            candidate = os.path.join(root, f)
            try:
                preview = pd.read_csv(candidate, nrows=5)
                # Check for IDRiD typical columns
                cols_lower = [c.lower() for c in preview.columns]
                if any('image' in c or 'id' in c for c in cols_lower) and any('grade' in c or 'retinopathy' in c or 'diag' in c for c in cols_lower):
                    idrid_csv = candidate
                    break
            except Exception:
                continue
    if idrid_csv:
        break

if idrid_csv:
    idrid_dir = os.path.dirname(idrid_csv)
    # Locate images folder
    for root, dirs, files in os.walk(os.path.dirname(idrid_dir) if 'label' in idrid_dir.lower() else idrid_dir):
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif')) for f in files):
            idrid_img_dir = root
            break

if idrid_csv and idrid_img_dir:
    print(f"✅ IDRiD Labels found: {idrid_csv}")
    print(f"✅ IDRiD Images found: {idrid_img_dir}")
    idrid_df_raw = pd.read_csv(idrid_csv)
    # Harmonize column names
    img_col = next(c for c in idrid_df_raw.columns if any(k in c.lower() for k in ['image', 'id']))
    diag_col = next(c for c in idrid_df_raw.columns if any(k in c.lower() for k in ['grade', 'retinopathy', 'diag']))
    idrid_df = idrid_df_raw[[img_col, diag_col]].dropna().copy()
    idrid_df.columns = ['id_code', 'diagnosis']
    idrid_df['diagnosis'] = idrid_df['diagnosis'].astype(int)
    print(f"Loaded {len(idrid_df)} verified IDRiD test samples.")
    
    idrid_dataset = RobustRetinaDataset(idrid_df, idrid_img_dir, transform=eval_transforms)
    idrid_loader = DataLoader(idrid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Evaluate
    idrid_logits_list, idrid_targets_list = [], []
    with torch.no_grad():
        for images, labels in idrid_loader:
            images = images.to(device)
            logits = model(images)
            idrid_logits_list.append(logits)
            idrid_targets_list.append(labels)
            
    idrid_logits = torch.cat(idrid_logits_list).to(device)
    idrid_targets = torch.cat(idrid_targets_list).cpu().numpy()
    
    # Calibrated probabilities
    idrid_calibrated_probs = torch.softmax(idrid_logits / optimal_T, dim=1).cpu().numpy()
    idrid_preds = np.argmax(idrid_calibrated_probs, axis=1)
    
    idrid_acc = accuracy_score(idrid_targets, idrid_preds)
    idrid_within_one = np.mean(np.abs(idrid_preds - idrid_targets) <= 1)
    idrid_qwk = cohen_kappa_score(idrid_targets, idrid_preds, weights='quadratic')
    idrid_rdr_targets = (idrid_targets >= 2).astype(int)
    idrid_rdr_probs = np.sum(idrid_calibrated_probs[:, 2:], axis=1)
    idrid_rdr_auc = roc_auc_score(idrid_rdr_targets, idrid_rdr_probs)
    idrid_ece = compute_ece(idrid_calibrated_probs, idrid_targets)
    
    print("\n================ IDRiD EXTERNAL TEST RESULTS ================")
    print(f"Exact Accuracy:          {idrid_acc*100:.2f}%")
    print(f"Within-1-Stage Accuracy: {idrid_within_one*100:.2f}%")
    print(f"Quadratic Weighted Kappa:{idrid_qwk:.4f}")
    print(f"Referable DR AUC:        {idrid_rdr_auc:.4f}")
    print(f"Calibrated ECE:          {idrid_ece*100:.2f}%")
    print("=============================================================")
else:
    print("ℹ️ IDRiD dataset not automatically detected in Kaggle inputs.")
    print("   To run IDRiD external testing: Add IDRiD to your Kaggle Notebook input.")


## 9. Publication-Quality Diagnostic Plots
Generates the core figures for your research paper:
1. Learning Curves (Loss & QWK across epochs)
2. Reliability Diagrams (Calibration ECE)
3. Normalized Confusion Matrix


In [ ]:
plt.figure(figsize=(16, 5))

# Plot 1: Training & Validation Curves
plt.subplot(1, 3, 1)
plt.plot(range(1, EPOCHS + 1), history['train_loss'], label='Train Loss', color='#2563EB', lw=2)
plt.plot(range(1, EPOCHS + 1), history['val_loss'], label='Val Loss', color='#DC2626', lw=2)
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Loss', fontsize=11)
plt.title('Loss Convergence Across 15 Epochs', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()

# Plot 2: QWK Trajectory
plt.subplot(1, 3, 2)
plt.plot(range(1, EPOCHS + 1), history['val_qwk'], label='Val QWK', color='#059669', lw=2.5, marker='o')
plt.axhline(best_val_qwk, color='#D97706', linestyle='--', label=f'Peak QWK ({best_val_qwk:.4f})')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Quadratic Weighted Kappa', fontsize=11)
plt.title('Validation QWK Trajectory', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()

# Plot 3: Held-out Test Confusion Matrix
cm = confusion_matrix(test_targets, test_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=['S0', 'S1', 'S2', 'S3', 'S4'],
            yticklabels=['S0', 'S1', 'S2', 'S3', 'S4'])
plt.xlabel('Predicted Severity', fontsize=11)
plt.ylabel('True Severity', fontsize=11)
plt.title(f'Normalized Test CM (QWK: {test_qwk:.3f})', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/publication_curves_and_cm.png', dpi=300)
plt.show()
print("Saved publication plot: /kaggle/working/publication_curves_and_cm.png")


## 10. Summary of Generated Paper Artifacts
Check that all models, calibration metrics, and figures are exported ready for publication tables.


In [ ]:
results_summary = {
    "model_backbone": "EfficientNet-B0",
    "epochs_trained": EPOCHS,
    "best_val_qwk": float(best_val_qwk),
    "optimal_temperature": float(optimal_T),
    "held_out_test_accuracy": float(test_acc),
    "held_out_test_within_one": float(within_one),
    "held_out_test_qwk": float(test_qwk),
    "held_out_test_ece": float(test_ece),
    "held_out_test_rdr_auc": float(rdr_auc)
}

with open('/kaggle/working/paper_experiment_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("📁 Generated Publication Artifacts:")
for f in os.listdir('/kaggle/working'):
    size_mb = os.path.getsize(os.path.join('/kaggle/working', f)) / (1024 * 1024)
    print(f"   - {f:<35} ({size_mb:.2f} MB)")

# -------------------------------------------------------------------------
# Package all artifacts for 1-click download
# -------------------------------------------------------------------------
import shutil
from IPython.display import FileLink

shutil.make_archive('/kaggle/working/paper_artifacts', 'zip', '/kaggle/working')
print('📦 Paper package created successfully:')
display(FileLink('paper_artifacts.zip'))
